### Simulasi Implementasi Integrasi Model CRNN dan RNN

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle

print("========================================")
print("MEMUAT MODEL INTEGRASI CRNN - RNN")
print("========================================")

# LOAD MODEL RNN

model_rnn = load_model(
    "hasil_modelling_rnn/rnn_suku_kata_model.keras"
)

with open(
    "hasil_modelling_rnn/input_tokenizer.pickle",
    "rb"
) as f:
    tokenizer_in = pickle.load(f)

with open(
    "hasil_modelling_rnn/target_tokenizer.pickle",
    "rb"
) as f:
    tokenizer_out = pickle.load(f)

reverse_out_index = {
    v:k
    for k,v in tokenizer_out.word_index.items()
}

print("✓ Model RNN berhasil dimuat")

# PANJANG SEQUENCE

max_len = 10

# FUNGSI REKONSTRUKSI

def rekonstruksi_kata(sequence_suku_kata):

    suku_list = sequence_suku_kata.split("-")

    seq = tokenizer_in.texts_to_sequences(
        [suku_list]
    )

    padded = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    prediksi = model_rnn.predict(
        padded,
        verbose=0
    )[0]

    hasil = []

    for timestep in prediksi:

        idx = np.argmax(timestep)

        if idx == 0:
            continue

        huruf = reverse_out_index.get(
            idx,
            ""
        )

        hasil.append(huruf)

    return "".join(hasil)

# SIMULASI OUTPUT CRNN

hasil_crnn = [
    "a-ba-di",
    "ja-nua-ri",
    "ma-ka-nan",
    "ka-mu"
]

print("\n=== SIMULASI INTEGRASI ===\n")

for suku_kata in hasil_crnn:

    hasil_kata = rekonstruksi_kata(
        suku_kata
    )

    print(
        f"CRNN : {suku_kata}"
    )

    print(
        f"RNN  : {hasil_kata}"
    )

    print("-"*40)

MEMUAT MODEL INTEGRASI CRNN - RNN
✓ Model RNN berhasil dimuat

=== SIMULASI INTEGRASI ===

CRNN : a-ba-di
RNN  : abadi
----------------------------------------
CRNN : ja-nua-ri
RNN  : januari
----------------------------------------
CRNN : ma-ka-nan
RNN  : madanan
----------------------------------------
CRNN : ka-mu
RNN  : kamu
----------------------------------------


### Validasi Pipeline Integrasi

In [32]:
import pandas as pd

df = pd.read_csv(
    "1_dataset_rnn/final_dataset.csv"
)

print(df.head())
print(len(df))

  Input_Suku_Kata Target_Kata_Utuh
0         a ba di            abadi
1           a bai             abai
2          a bang            abang
3            abdi             abdi
4           abjad            abjad
1864


In [33]:
df_uji = df.head(50).copy()

print(df_uji.head())

  Input_Suku_Kata Target_Kata_Utuh
0         a ba di            abadi
1           a bai             abai
2          a bang            abang
3            abdi             abdi
4           abjad            abjad


In [34]:
hasil_integrasi = []

for _, row in df_uji.iterrows():

    input_suku = row["Input_Suku_Kata"]
    target = row["Target_Kata_Utuh"]

    prediksi = rekonstruksi_kata(
        input_suku.replace(" ", "-")
    )

    status = (
        "BENAR"
        if prediksi == target
        else "SALAH"
    )

    hasil_integrasi.append({
        "Input_Suku_Kata": input_suku,
        "Target": target,
        "Prediksi": prediksi,
        "Status": status
    })

In [35]:
hasil_df = pd.DataFrame(
    hasil_integrasi
)

print(hasil_df)

   Input_Suku_Kata    Target  Prediksi Status
0          a ba di     abadi     abadi  BENAR
1            a bai      abai      abai  BENAR
2           a bang     abang      aban  SALAH
3             abdi      abdi      abdi  BENAR
4            abjad     abjad     abjad  BENAR
5            a bon      abon      abon  BENAR
6          a borsi    aborsi    aborsi  BENAR
7            absen     absen     absen  BENAR
8         abso lut   absolut   absolut  BENAR
9          abstrak   abstrak   abstrak  BENAR
10            a bu       abu       abu  BENAR
11           a cak      acak      acak  BENAR
12           a cam      acam      acam  BENAR
13         a ca ra     acara     acara  BENAR
14            a cu       acu       acu  BENAR
15          a cuan     acuan     acuan  BENAR
16           a cur      acur      acur  BENAR
17            a da       ada       ada  BENAR
18           a dab      adab      adab  BENAR
19           a dat      adat      adat  BENAR
20      a di da ya   adidaya   adi

In [36]:
jumlah_benar = (
    hasil_df["Status"]
    == "BENAR"
).sum()

jumlah_data = len(
    hasil_df
)

akurasi = (
    jumlah_benar
    /
    jumlah_data
) * 100

print("\n=== HASIL INTEGRASI ===")
print(f"Jumlah Data  : {jumlah_data}")
print(f"Benar        : {jumlah_benar}")
print(f"Salah        : {jumlah_data-jumlah_benar}")
print(f"Akurasi      : {akurasi:.2f}%")


=== HASIL INTEGRASI ===
Jumlah Data  : 50
Benar        : 43
Salah        : 7
Akurasi      : 86.00%


### Analisis Error dan Keterbatasan Sistem

In [37]:
salah_df = hasil_df[
    hasil_df["Status"] == "SALAH"
]

print(salah_df)
print("\nJumlah Error:")
print(len(salah_df))

   Input_Suku_Kata    Target  Prediksi Status
2           a bang     abang      aban  SALAH
27           adhem     adhem      adhe  SALAH
29     a fi ni tas  afinitas  akisisis  SALAH
30      a firma si  afirmasi    afosii  SALAH
35         a ga ma     agama      agam  SALAH
44           a kad      akad      agom  SALAH
49           akhir     akhir    eespam  SALAH

Jumlah Error:
7


### Segmentasi Huruf Menjadi Suku Kata

In [38]:
def segmentasi_suku_kata(kata):

    kata = kata.lower()

    vokal = "aiueo"

    hasil = []
    buffer = ""

    for huruf in kata:

        buffer += huruf

        if huruf in vokal:
            hasil.append(buffer)
            buffer = ""

    if buffer:
        hasil[-1] += buffer

    return "-".join(hasil)

In [39]:
hasil_segmentasi = []

for _, row in df.iterrows():

    target_suku = row["Input_Suku_Kata"]
    kata_utuh = row["Target_Kata_Utuh"]

    # format dataset:
    # a ba di
    # menjadi
    # a-ba-di

    target_suku = target_suku.replace(" ", "-")

    hasil_prediksi = segmentasi_suku_kata(
        kata_utuh
    )

    status = (
        "BENAR"
        if hasil_prediksi == target_suku
        else "SALAH"
    )

    hasil_segmentasi.append({
        "Kata_Utuh": kata_utuh,
        "Target_Suku_Kata": target_suku,
        "Prediksi_Segmentasi": hasil_prediksi,
        "Status": status
    })

In [40]:
import pandas as pd

df_segmentasi = pd.DataFrame(
    hasil_segmentasi
)

df_segmentasi.head(50)

,Kata_Utuh,Target_Suku_Kata,Prediksi_Segmentasi,Status
0,abadi,a-ba-di,a-ba-di,BENAR
1,abai,a-bai,a-ba-i,SALAH
2,abang,a-bang,a-bang,BENAR
3,abdi,abdi,a-bdi,SALAH
4,abjad,abjad,a-bjad,SALAH
5,abon,a-bon,a-bon,BENAR
6,aborsi,a-borsi,a-bo-rsi,SALAH
7,absen,absen,a-bsen,SALAH
8,absolut,abso-lut,a-bso-lut,SALAH
9,abstrak,abstrak,a-bstrak,SALAH


In [41]:
jumlah_data = len(df_segmentasi)

jumlah_benar = len(
    df_segmentasi[
        df_segmentasi["Status"] == "BENAR"
    ]
)

jumlah_salah = len(
    df_segmentasi[
        df_segmentasi["Status"] == "SALAH"
    ]
)

akurasi = (
    jumlah_benar /
    jumlah_data
) * 100

print("\n=== VALIDASI MODUL SEGMENTASI ===")
print(f"Jumlah Data : {jumlah_data}")
print(f"Benar       : {jumlah_benar}")
print(f"Salah       : {jumlah_salah}")
print(f"Akurasi     : {akurasi:.2f}%")


=== VALIDASI MODUL SEGMENTASI ===
Jumlah Data : 1864
Benar       : 1281
Salah       : 583
Akurasi     : 68.72%


In [42]:
error_segmentasi = df_segmentasi[
    df_segmentasi["Status"] == "SALAH"
]

print(
    "\nJumlah Error :",
    len(error_segmentasi)
)

error_segmentasi.head(50)


Jumlah Error : 583


,Kata_Utuh,Target_Suku_Kata,Prediksi_Segmentasi,Status
1,abai,a-bai,a-ba-i,SALAH
3,abdi,abdi,a-bdi,SALAH
4,abjad,abjad,a-bjad,SALAH
6,aborsi,a-borsi,a-bo-rsi,SALAH
7,absen,absen,a-bsen,SALAH
8,absolut,abso-lut,a-bso-lut,SALAH
9,abstrak,abstrak,a-bstrak,SALAH
15,acuan,a-cuan,a-cu-an,SALAH
22,adopsi,a-dopsi,a-do-psi,SALAH
24,aduan,a-duan,a-du-an,SALAH


In [43]:
print(segmentasi_suku_kata("ABADI"))
print(segmentasi_suku_kata("KAMU"))
print(segmentasi_suku_kata("JANUARI"))
print(segmentasi_suku_kata("MAKANAN"))

a-ba-di
ka-mu
ja-nu-a-ri
ma-ka-nan


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report
import numpy as np

# CELL 15.C: VISUALISASI & SIMPAN BAR CHART 
print(" [PROSES] Menyusun grafik hasil evaluasi dan menyimpan gambar...")

# 1. Ambil data Target dan Prediksi dari hasil pengujian model RNN sebelumnya
y_true_list = hasil_df["Target"].tolist()
y_pred_list = hasil_df["Prediksi"].tolist()

# 2. Cetak Classification Report (Opsional untuk memastikan data valid)
# print(classification_report(y_true_list, y_pred_list, zero_division=0))

# 3. Contoh daftar kata uji untuk ditampilkan
hasil_crnn_huruf = [
    "ABADI",
    "KAMU",
    "JANUARI",
    "MAKANAN"
]

for kata in hasil_crnn_huruf:
    hasil_segmentasi = segmentasi_suku_kata(kata)

    print(f"Huruf CRNN : {kata}")
    print(f"Suku Kata  : {hasil_segmentasi}")
    print("-"*40)

 [PROSES] Menyusun grafik hasil evaluasi dan menyimpan gambar...
Huruf CRNN : ABADI
Suku Kata  : a-ba-di
----------------------------------------
Huruf CRNN : KAMU
Suku Kata  : ka-mu
----------------------------------------
Huruf CRNN : JANUARI
Suku Kata  : ja-nu-a-ri
----------------------------------------
Huruf CRNN : MAKANAN
Suku Kata  : ma-ka-nan
----------------------------------------


### Simulasi Integrasi

In [ ]:
# CONTOH INTEGRASI DENGAN OUTPUT CRNN NYATA

# 1. Misalkan ini adalah string huruf hasil pembacaan model CRNN 
huruf_dari_crnn = "KACUNG" 

# 2. Ubah dulu huruf dari CRNN menjadi format suku kata menggunakan fungsi segmentasi
suku_kata_dari_crnn = segmentasi_suku_kata(huruf_dari_crnn)
# Hasilnya akan menjadi: "a-ba-di"

# 3. Masukkan ke fungsi rekonstruksi RNN yang sudah Anda buat
hasil_akhir_rnn = rekonstruksi_kata(suku_kata_dari_crnn)

print(f"Input Huruf CRNN : {huruf_dari_crnn}")
print(f"Hasil Suku Kata  : {suku_kata_dari_crnn}")
print(f"Hasil Akhir RNN  : {hasil_akhir_rnn}")

Input Huruf CRNN : KACUNG
Hasil Suku Kata  : ka-cung
Hasil Akhir RNN  : kacom
